In [5]:
# ==========================================
# VisionBench - Baseline 8: FastFlow (2D Normalizing Flows)
# ==========================================

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from torch.utils.data import DataLoader, Dataset
import torchvision.models as models
import numpy as np
from sklearn.metrics import roc_auc_score

# 1. Environment Setup & Anomalib Light Engine
!pip install 'anomalib[full]'==1.0.0 --quiet

from anomalib.models import Fastflow
from anomalib.data.mvtec_ad import MVTec
from anomalib.engine import Engine

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

# 2. FastFlow Model Initialization (ResNet18 Backbone)
# Normalizing flow maps feature representations directly into a normal distribution
model = Fastflow(
    backbone="resnet18",
    flow_steps=8,
    pre_trained=True
)

# 3. MVTec Dataset Module Setup (Fast execution mode)
datamodule = MVTec(
    category="bottle",  # Can loop over all 15 MVTec AD categories
    image_size=(256, 256),
    train_batch_size=16,
    eval_batch_size=16
)

# 4. Engine Trainer Execution
engine = Engine(max_epochs=5, accelerator="gpu", devices=1)

print("\n--- Starting FastFlow Baseline 8 Training ---")
engine.fit(model=model, datamodule=datamodule)

print("\n--- Evaluating Baseline 8 Metrics ---")
test_results = engine.test(model=model, datamodule=datamodule)
print(test_results)

  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


ModuleNotFoundError: No module named 'anomalib.data.mvtec_ad'

In [11]:
# ==========================================
# VisionBench - Baseline 8: FastFlow (2D Normalizing Flow)
# Pure PyTorch Standalone Implementation
# ==========================================

import os
import tarfile
import urllib.request
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as T
import torchvision.models as models
from PIL import Image
from sklearn.metrics import roc_auc_score

# ------------------------------------------
# 1. Automatic Dataset Setup (MVTec AD Bottle)
# ------------------------------------------
# ------------------------------------------
# 1. Automatic Dataset Setup (HuggingFace Stable Mirror)
# ------------------------------------------
# ------------------------------------------
# 1. Automatic Dataset Setup (Bottle Only)
# ------------------------------------------
# ------------------------------------------
# 1. Automatic Dataset Setup (KaggleHub)
# ------------------------------------------
import kagglehub

print("Downloading MVTec AD dataset via KaggleHub...")
# Downloads and extracts the dataset reliably in Colab
path = kagglehub.dataset_download("ipythonx/mvtec-ad")

# Set paths to point to the local dataset cache
DATA_DIR = Path(path)
CATEGORY = "bottle"
CAT_DIR = DATA_DIR / CATEGORY

print(f"Dataset ready at: {CAT_DIR}")
# ------------------------------------------
# 2. PyTorch Dataset Loader Definition
# ------------------------------------------
class MVTecDataset(Dataset):
    def __init__(self, root_dir, category, split="train", transform=None):
        self.transform = transform
        self.image_paths = []
        self.labels = []  # 0 for normal, 1 for anomaly

        cat_path = root_dir / category / split
        if split == "train":
            good_dir = cat_path / "good"
            for img in good_dir.glob("*.png"):
                self.image_paths.append(img)
                self.labels.append(0)
        else:
            for sub_dir in cat_path.iterdir():
                if sub_dir.is_dir():
                    is_good = sub_dir.name == "good"
                    for img in sub_dir.glob("*.png"):
                        self.image_paths.append(img)
                        self.labels.append(0 if is_good else 1)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, self.labels[idx]

# Datasets and DataLoaders
img_transform = T.Compose([
    T.Resize((256, 256)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_ds = MVTecDataset(DATA_DIR, CATEGORY, split="train", transform=img_transform)
test_ds = MVTecDataset(DATA_DIR, CATEGORY, split="test", transform=img_transform)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False)

# ------------------------------------------
# 3. FastFlow Architecture Definition
# ------------------------------------------
class FlowSubstep(nn.Module):
    """2D Convolutional Normalizing Flow Transformation Block"""
    def __init__(self, in_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels // 2, in_channels, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels, in_channels // 2, kernel_size=3, padding=1)
        )

    def forward(self, x):
        x1, x2 = x.chunk(2, dim=1)
        y1 = x1
        y2 = x2 + self.net(x1)
        return torch.cat([y1, y2], dim=1)

class FastFlowModel(nn.Module):
    """FastFlow using Frozen ResNet18 Feature Extractor"""
    def __init__(self):
        super().__init__()
        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.feature_extractor = nn.Sequential(*list(resnet.children())[:-3])
        for param in self.feature_extractor.parameters():
            param.requires_grad = False  # Freeze feature backbone

        self.flow = nn.Sequential(
            FlowSubstep(256),
            FlowSubstep(256),
            FlowSubstep(256)
        )

    def forward(self, x):
        features = self.feature_extractor(x)
        z = self.flow(features)
        # Compute Negative Log-Likelihood (NLL) Loss
        loss = 0.5 * torch.mean(z ** 2)
        return loss, z

# ------------------------------------------
# 4. Model Training & Validation Loop
# ------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = FastFlowModel().to(device)
optimizer = torch.optim.Adam(model.flow.parameters(), lr=1e-3)

epochs = 5
print(f"\n--- Starting Baseline 8 Training ({CATEGORY.upper()}) on {device} ---")

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for images, _ in train_loader:
        images = images.to(device)
        optimizer.zero_grad()

        loss, _ = model(images)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}] | Loss (NLL): {avg_loss:.4f}")

# ------------------------------------------
# 5. Evaluation & AUROC Metric Computation
# ------------------------------------------
print("\n--- Evaluating Baseline 8 Anomaly Scores ---")
model.eval()
all_scores = []
all_targets = []

with torch.no_grad():
    for images, targets in test_loader:
        images = images.to(device)
        _, z = model(images)

        # Calculate anomaly score per image based on flow deviation
        scores = torch.mean(z ** 2, dim=[1, 2, 3]).cpu().numpy()
        all_scores.extend(scores)
        all_targets.extend(targets)

auroc = roc_auc_score(all_targets, all_scores)
print("==========================================")
print(f" Baseline 8 (FastFlow) Results [{CATEGORY.upper()}]:")
print(f" Final Test Image AUROC: {auroc:.4f}")
print("==========================================")

Using Colab cache for faster access to the 'mvtec-ad' dataset.
Dataset ready at: /kaggle/input/mvtec-ad/bottle
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 139MB/s]



--- Starting Baseline 8 Training (BOTTLE) on cuda ---
Epoch [1/5] | Loss (NLL): 0.0317
Epoch [2/5] | Loss (NLL): 0.0253
Epoch [3/5] | Loss (NLL): 0.0225
Epoch [4/5] | Loss (NLL): 0.0209
Epoch [5/5] | Loss (NLL): 0.0199

--- Evaluating Baseline 8 Anomaly Scores ---
 Baseline 8 (FastFlow) Results [BOTTLE]:
 Final Test Image AUROC: 0.9786


In [12]:
# ==============================================================================
# VisionBench - Baseline 8: FastFlow (2D Normalizing Flow)
# Pure PyTorch Standalone Implementation (Full 15-Category Benchmark Matrix)
# ==============================================================================

import os
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms as T
import torchvision.models as models
from torch.utils.data import DataLoader, Dataset
from PIL import Image
from sklearn.metrics import roc_auc_score
import kagglehub

# ------------------------------------------------------------------------------
# 1. Automatic Dataset Setup via KaggleHub
# ------------------------------------------------------------------------------
print("--- Step 1: Loading MVTec AD Dataset ---")
kaggle_path = kagglehub.dataset_download("ipythonx/mvtec-ad")
DATA_DIR = Path(kaggle_path)
print(f"Dataset cached at: {DATA_DIR}\n")

# ------------------------------------------------------------------------------
# 2. PyTorch Dataset Loader Definition
# ------------------------------------------------------------------------------
class MVTecDataset(Dataset):
    def __init__(self, root_dir, category, split="train", transform=None):
        self.transform = transform
        self.image_paths = []
        self.labels = []  # 0 for normal, 1 for anomaly

        cat_path = root_dir / category / split
        if split == "train":
            good_dir = cat_path / "good"
            for img in good_dir.glob("*.png"):
                self.image_paths.append(img)
                self.labels.append(0)
        else:
            for sub_dir in cat_path.iterdir():
                if sub_dir.is_dir():
                    is_good = (sub_dir.name == "good")
                    for img in sub_dir.glob("*.png"):
                        self.image_paths.append(img)
                        self.labels.append(0 if is_good else 1)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, self.labels[idx]

img_transform = T.Compose([
    T.Resize((256, 256)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ------------------------------------------------------------------------------
# 3. FastFlow Architecture Definition
# ------------------------------------------------------------------------------
class FlowSubstep(nn.Module):
    """2D Convolutional Normalizing Flow Transformation Block"""
    def __init__(self, in_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels // 2, in_channels, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels, in_channels // 2, kernel_size=3, padding=1)
        )

    def forward(self, x):
        x1, x2 = x.chunk(2, dim=1)
        y1 = x1
        y2 = x2 + self.net(x1)
        return torch.cat([y1, y2], dim=1)

class FastFlowModel(nn.Module):
    """FastFlow using Frozen ResNet18 Feature Extractor"""
    def __init__(self):
        super().__init__()
        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.feature_extractor = nn.Sequential(*list(resnet.children())[:-3])
        for param in self.feature_extractor.parameters():
            param.requires_grad = False  # Freeze feature backbone

        self.flow = nn.Sequential(
            FlowSubstep(256),
            FlowSubstep(256),
            FlowSubstep(256)
        )

    def forward(self, x):
        features = self.feature_extractor(x)
        z = self.flow(features)
        # Compute Negative Log-Likelihood (NLL) Loss
        loss = 0.5 * torch.mean(z ** 2)
        return loss, z

# ------------------------------------------------------------------------------
# 4. Multi-Category Execution Loop across 15 MVTec AD Categories
# ------------------------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running Baseline 8 on Execution Device: {device}\n")

CATEGORIES = [
    "bottle", "cable", "capsule", "carpet", "grid",
    "hazelnut", "leather", "metal_nut", "pill", "screw",
    "tile", "toothbrush", "transistor", "wood", "zipper"
]

results = {}

print("--- Step 2: Training & Evaluating All 15 Categories ---")

for cat in CATEGORIES:
    print(f"\nProcessing Category: [{cat.upper()}]")

    # Load Datasets
    train_ds = MVTecDataset(DATA_DIR, cat, split="train", transform=img_transform)
    test_ds = MVTecDataset(DATA_DIR, cat, split="test", transform=img_transform)

    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=16, shuffle=False)

    # Initialize Model & Optimizer
    model = FastFlowModel().to(device)
    optimizer = torch.optim.Adam(model.flow.parameters(), lr=1e-3)

    # Fast 3-Epoch Training Loop
    epochs = 3
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for images, _ in train_loader:
            images = images.to(device)
            optimizer.zero_grad()
            loss, _ = model(images)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        avg_loss = running_loss / len(train_loader)
        print(f"  Epoch [{epoch+1}/{epochs}] | NLL Loss: {avg_loss:.4f}")

    # Evaluation
    model.eval()
    all_scores, all_targets = [], []
    with torch.no_grad():
        for images, targets in test_loader:
            images = images.to(device)
            _, z = model(images)
            scores = torch.mean(z ** 2, dim=[1, 2, 3]).cpu().numpy()
            all_scores.extend(scores)
            all_targets.extend(targets)

    cat_auroc = roc_auc_score(all_targets, all_scores)
    results[cat] = cat_auroc
    print(f"  -> Result: {cat.upper()} Image AUROC = {cat_auroc:.4f}")

# ------------------------------------------------------------------------------
# 5. Final Benchmark Metrics Summary
# ------------------------------------------------------------------------------
print("\n==================================================")
print("  VISIONBENCH: BASELINE 8 (FASTFLOW) RESULTS")
print("==================================================")
for cat, score in results.items():
    print(f"  {cat:<15}: {score:.4f}")

mean_auroc = np.mean(list(results.values()))
print("--------------------------------------------------")
print(f"  MEAN AUROC (15 Categories): {mean_auroc:.4f}")
print("==================================================")

--- Step 1: Loading MVTec AD Dataset ---
Using Colab cache for faster access to the 'mvtec-ad' dataset.
Dataset cached at: /kaggle/input/mvtec-ad

Running Baseline 8 on Execution Device: cuda

--- Step 2: Training & Evaluating All 15 Categories ---

Processing Category: [BOTTLE]
  Epoch [1/3] | NLL Loss: 0.0314
  Epoch [2/3] | NLL Loss: 0.0250
  Epoch [3/3] | NLL Loss: 0.0223
  -> Result: BOTTLE Image AUROC = 0.9603

Processing Category: [CABLE]
  Epoch [1/3] | NLL Loss: 0.0317
  Epoch [2/3] | NLL Loss: 0.0274
  Epoch [3/3] | NLL Loss: 0.0253
  -> Result: CABLE Image AUROC = 0.6874

Processing Category: [CAPSULE]
  Epoch [1/3] | NLL Loss: 0.0285
  Epoch [2/3] | NLL Loss: 0.0239
  Epoch [3/3] | NLL Loss: 0.0212
  -> Result: CAPSULE Image AUROC = 0.4787

Processing Category: [CARPET]
  Epoch [1/3] | NLL Loss: 0.0272
  Epoch [2/3] | NLL Loss: 0.0226
  Epoch [3/3] | NLL Loss: 0.0218
  -> Result: CARPET Image AUROC = 0.7737

Processing Category: [GRID]
  Epoch [1/3] | NLL Loss: 0.0258
  Epo